In [ ]:
print("I am ready")

In [ ]:
!pip install -q sentence-transformers faiss-cpu \
 rank-bm25 openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.0 MB/s eta 0:00:00


In [ ]:
import os
MANUAL_PATH="product_manual_1.md"
if not os.path.exists(MANUAL_PATH):
  from google.colab import files
  print("choose product_manual.md")
  uploaded=files.upload()
  MANUAL_PATH=list(uploaded.keys())[0]
with open (MANUAL_PATH,encoding="utf-8") as f:
  MANUAL_TEXT=f.read()
print("Loaded", MANUAL_PATH)
print(len(MANUAL_TEXT),"characters")

Loaded product_manual_1.md
9238 characters


In [ ]:
def chunk_fixed(text,size=600,overlap=100):
  if overlap>=size:
    raise ValueError("overlap must be smaller than size")
  chunks,start=[],0
  while start<len(text):
    chunks.append(text[start:start +size])
    start+=size-overlap
  return chunks

fixed_chunks=chunk_fixed(MANUAL_TEXT)
print(f"Fixed size gave {len(fixed_chunks)} chunks.\n")
print("--- chunk3, notice where it starts and ends ---")
print(repr(fixed_chunks[3][:200]),"...")

Fixed size gave 19 chunks.

--- chunk3, notice where it starts and ends ---
'## Section 3: Getting Started and Pairing\n\nTo turn on the Aurora M2, press and hold the power button for two seconds. A white light indicates the headphones are on.\n\nTo pair for the first time:\n\n1. Wi' ...


In [ ]:
def chunk_recursive(text,max_size=600):
  paras=[p.strip() for p in text.split("\n\n")if p.strip()]
  chunks,current=[],""
  for p in paras:
    if current and len(current)+len(p)+2 > max_size:
      chunks.append(current)
      current=p
    else:
      current=(current+"\n\n"+p) if current else p
  if current :
    chunks.append(current)
  return chunks


recursive_chunks=chunk_recursive(MANUAL_TEXT)
print(f"Recursive gave {len(recursive_chunks)} chunks.\n")
print("--- chunk 3, whole paragraphs---")
print(recursive_chunks[3][:250],"...")

Recursive gave 19 chunks.

--- chunk 3, whole paragraphs---
The Aurora M2 supports multipoint connection, meaning it can be connected to two devices at the same time, for example a laptop and a phone.

---

## Section 3: Getting Started and Pairing

To turn on the Aurora M2, press and hold the power button fo ...


In [ ]:
import re
def chunk_by_section(text):
  parts =re.split(r"\n(?=##)",text)
  out=[]
  for part in parts:
    part = part.strip()
    if not part :
      continue
    heading=part.split("\n")[0].replace("#","").strip()
    out.append({"text":part,"section":heading})
  return out

section_chunks= chunk_by_section(MANUAL_TEXT)
print(f"Section based gave {len(section_chunks)} chunks:\n")
for c in section_chunks:
  print(f"{len(c['text']):5d} chars | {c['section']}")

Section based gave 12 chunks:

  243 chars | Aurora M2 Wireless Headphones: User Manual
  462 chars | Section 1: In the Box
  789 chars | Section 2: Specifications
  942 chars | Section 3: Getting Started and Pairing
  619 chars | Section 4: Controls
  733 chars | Section 5: Noise Cancellation and Transparency
  714 chars | Section 6: Firmware Updates
 1477 chars | Section 7: Troubleshooting
  580 chars | Section 8: Care and Maintenance
  662 chars | Section 9: Warranty
  817 chars | Section 10: Returns and Refunds
 1177 chars | Section 11: Frequently Asked Questions


In [ ]:
def tag_type(section):
  s=section.lower()
  if "troubleshoot" in s :
    return "troubleshooting"
  if "warrant" in s or "return" in s or "refund" in s :
    return "policy"
  if "spec" in s :
    return "specs"
  if "faq" in s or "question" in s :
    return "faq"
  return "general"

chunks=[]
for i,c in enumerate (section_chunks):
  chunks.append({
      "id": f"chunk_{i:02d}",
      "text": c["text"],
      "section": c["section"],
      "content_type": tag_type(c["section"]),
  })

print (f"{len(chunks)} chunks with metadata :\n")
for c in chunks :
  print (f"{c['id']} | {c['content_type']:15s} | {c['section']}")

12 chunks with metadata :

chunk_00 | general         | Aurora M2 Wireless Headphones: User Manual
chunk_01 | general         | Section 1: In the Box
chunk_02 | specs           | Section 2: Specifications
chunk_03 | general         | Section 3: Getting Started and Pairing
chunk_04 | general         | Section 4: Controls
chunk_05 | general         | Section 5: Noise Cancellation and Transparency
chunk_06 | general         | Section 6: Firmware Updates
chunk_07 | troubleshooting | Section 7: Troubleshooting
chunk_08 | general         | Section 8: Care and Maintenance
chunk_09 | policy          | Section 9: Warranty
chunk_10 | policy          | Section 10: Returns and Refunds
chunk_11 | faq             | Section 11: Frequently Asked Questions


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-mpnet-base-v2")

# normalize_embeddings=True makes every vector length 1,
# so the dot product between two vectors IS their cosine similarity.
chunk_vectors = embed_model.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("\nShape:", chunk_vectors.shape)
print(f"-> {chunk_vectors.shape[0]} chunks, each {chunk_vectors.shape[1]} numbers.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Shape: (12, 768)
-> 12 chunks, each 768 numbers.


In [ ]:
pairs = embed_model.encode([
    "How long does the battery last?",              # 0
    "What is the playback time on a full charge?",  # 1 same meaning, different words
    "What colour options are available?",           # 2 unrelated
], normalize_embeddings=True)

print("battery vs playback time :", round(float(pairs[0] @ pairs[1]), 3), " <- high,same meaning")
print("battery vs colour        :", round(float(pairs[0] @ pairs[2]), 3), " <- low,unrelated")

battery vs playback time : 0.596  <- high,same meaning
battery vs colour        : 0.086  <- low,unrelated


In [ ]:
import faiss
index= faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors.astype("float32"))

print("Index holds ", index.ntotal,"vectors.")

Index holds  12 vectors.


In [ ]:

def vector_search(query, k=3):
    q = embed_model.encode(query, normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(np.array([q]), k)
    return [
        {"chunk": chunks[i], "score": float(s)}
        for s, i in zip(scores[0], idxs[0])
    ]


for r in vector_search("how long can I listen before recharging?"):
    print(f"  {r['score']:.3f} | {r['chunk']['section']}")

  0.533 | Section 2: Specifications
  0.381 | Section 11: Frequently Asked Questions
  0.341 | Section 8: Care and Maintenance


In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    """Lowercase; keep alphanumerics and internal hyphens.
    'Error E-03:' -> ['error', 'e-03']   (not 'e-03:')"""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def keyword_search(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [{"chunk": chunks[i], "score": float(scores[i])} for i in top]

# Proof the tokenizer works: a bare error code must score above zero.
print("Score for the bare token 'E-03':",
      round(float(max(bm25.get_scores(tokenize("E-03")))), 2), "\n")

print("KEYWORD search for 'error E-03 no sound':")
for r in keyword_search("error E-03 no sound"):
    print(f"  {r['score']:6.2f}  |  {r['chunk']['section']}")

print("\nVECTOR search for the same thing (less certain about the code):")
for r in vector_search("error E-03 no sound"):
    print(f"  {r['score']:6.3f}  |  {r['chunk']['section']}")

Score for the bare token 'E-03': 1.42 

KEYWORD search for 'error E-03 no sound':
    7.02  |  Section 7: Troubleshooting
    1.48  |  Section 5: Noise Cancellation and Transparency
    1.45  |  Section 6: Firmware Updates

VECTOR search for the same thing (less certain about the code):
   0.433  |  Section 7: Troubleshooting
   0.309  |  Section 6: Firmware Updates
   0.262  |  Section 5: Noise Cancellation and Transparency


In [ ]:
def hybrid_search(query, k=3, pool=6):
    vec = vector_search(query, pool)
    key = keyword_search(query, pool)

    rrf = {}
    for ranked in (vec, key):
        for rank, r in enumerate(ranked):     # rank 0, 1, 2, ...
            cid = r["chunk"]["id"]
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (60 + rank + 1)

    by_id = {c["id"]: c for c in chunks}
    best = sorted(rrf.items(), key=lambda kv: kv[1], reverse=True)[:k]
    return [{"chunk": by_id[cid], "score": s} for cid, s in best]

print("HYBRID for 'error E-03 no sound':")
for r in hybrid_search("error E-03 no sound"):
    print(f"  {r['score']:.4f}  |  {r['chunk']['section']}")

print("\nHYBRID for 'can I get my money back':")
for r in hybrid_search("can I get my money back"):
    print(f"  {r['score']:.4f}  |  {r['chunk']['section']}")


HYBRID for 'error E-03 no sound':
  0.0328  |  Section 7: Troubleshooting
  0.0320  |  Section 6: Firmware Updates
  0.0320  |  Section 5: Noise Cancellation and Transparency

HYBRID for 'can I get my money back':
  0.0325  |  Section 10: Returns and Refunds
  0.0320  |  Section 11: Frequently Asked Questions
  0.0315  |  Section 8: Care and Maintenance


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, k=3):
    """Score (query, chunk) pairs together and keep the best k."""
    pairs = [[query, c["chunk"]["text"]] for c in candidates]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1][:k]
    return [{"chunk": candidates[i]["chunk"], "score": float(scores[i])} for i in order]

def search(query, k=3, pool=6):
    """Full retrieval: hybrid search for a shortlist, then re-rank it."""
    shortlist = hybrid_search(query, k=pool, pool=pool)
    return rerank(query, shortlist, k=k)

q = "I opened the box 20 days ago, can I still return them?"
print("Before re-ranking (hybrid order):")
for r in hybrid_search(q, k=4):
    print(f"   {r['chunk']['section']}")

print("\nAfter re-ranking:")
for r in search(q, k=3):
    print(f"  {r['score']:7.2f}  |  {r['chunk']['section']}")



config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Before re-ranking (hybrid order):
   Section 10: Returns and Refunds
   Section 8: Care and Maintenance
   Section 6: Firmware Updates
   Section 11: Frequently Asked Questions

After re-ranking:
    -3.68  |  Section 10: Returns and Refunds
   -10.24  |  Section 1: In the Box
   -11.29  |  Section 6: Firmware Updates


In [ ]:
def filtered_search(query, content_type=None, k=3):
    keep = [i for i, c in enumerate(chunks)
            if content_type is None or c["content_type"] == content_type]
    if not keep:
        return []
    q = embed_model.encode(query, normalize_embeddings=True)
    sims = chunk_vectors[keep] @ q                    # reuse existing vectors
    order = np.argsort(sims)[::-1][:k]
    return [{"chunk": chunks[keep[j]], "score": float(sims[j])} for j in order]

print("Unfiltered, searches everything:")
for r in vector_search("what is the fee", k=3):
    print(f"  {r['chunk']['content_type']:15s} | {r['chunk']['section']}")

print("\nFiltered to policy sections only:")
for r in filtered_search("what is the fee", content_type="policy", k=3):
    print(f"  {r['chunk']['content_type']:15s} | {r['chunk']['section']}")


Unfiltered, searches everything:
  policy          | Section 10: Returns and Refunds
  specs           | Section 2: Specifications
  general         | Section 1: In the Box

Filtered to policy sections only:
  policy          | Section 10: Returns and Refunds
  policy          | Section 9: Warranty


In [ ]:
from getpass import getpass
from openai import OpenAI

LAGUNA_KEY = "sky_Knh9X5Q5.NGaFmzew8807p3FCh4PYAP3zjLh3eYeI"

client = OpenAI(api_key=LAGUNA_KEY, base_url="https://inference.poolside.ai/v1")
MODEL = "poolside/laguna-s-2.1"
print("Client created.")

def ask_llm(prompt, temperature=0.0, max_tokens=400):
    """Send a prompt to the model, return the text answer."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content

print(ask_llm("Reply with exactly: RAG workshop is ready."))

ANSWER_PROMPT = """Answer using ONLY the context below. Don't just answer, also append the reason, context and enough information.
If the answer is not in the context, reply exactly:
"I could not find this in the manual."
Cite the section you used in square brackets, like [Section 9: Warranty].

Context:
{context}

Question: {question}
Answer:"""

def rag_answer(question, k=3):
    hits = search(question, k=k)
    context = "\n\n---\n\n".join(
        f"[{h['chunk']['section']}]\n{h['chunk']['text']}" for h in hits
    )
    prompt = ANSWER_PROMPT.format(context=context, question=question)
    return ask_llm(prompt), hits

answer, hits = rag_answer("How long is the warranty?")
print("ANSWER:", answer)
print("\nRetrieved from:", [h["chunk"]["section"] for h in hits])

Client created.
RAG workshop is ready.
ANSWER: The warranty for the Aurora M2 is 24 months from the date of purchase. [Section 9: Warranty]

Retrieved from: ['Section 9: Warranty', 'Section 10: Returns and Refunds', 'Section 2: Specifications']


In [ ]:
questions = [
    "What is the battery life with noise cancellation on?",
    "What does error E-04 mean?",
    "I opened the box 20 days ago and they won't pair. Can I still get a refund?",
    "What is the wifi password for the headphones?",
]

for q in questions:
    ans, hits = rag_answer(q)
    print("Q:", q)
    print("A:", ans)
    print("   sources:", [h["chunk"]["section"] for h in hits])
    print("-" * 70)

Q: What is the battery life with noise cancellation on?
A: The battery life with noise cancellation on is up to 28 hours. This is confirmed in [Section 2: Specifications], which states, "Battery life: up to 40 hours with active noise cancellation off, up to 28 hours with noise cancellation on." Additionally, [Section 5: Noise Cancellation and Transparency] reiterates this by noting, "Turning noise cancellation off increases battery life from 28 hours to 40 hours," implying that 28 hours is the standard battery life when noise cancellation is active.
   sources: ['Section 5: Noise Cancellation and Transparency', 'Section 2: Specifications', 'Section 4: Controls']
----------------------------------------------------------------------
Q: What does error E-04 mean?
A: Answer: Error E-04 means the headphones are "Keeps disconnecting." This is usually caused by distance or interference. Stay within 10 metres of the connected device and away from strong Wi-Fi routers or microwave ovens. If it